# Public Curves And Surfaces

This small notebook shows the current first-layer public imports for fitted
curves and volatility surfaces.


In [ ]:
from decimal import Decimal

from fuggers_py import Date
from fuggers_py.curves import (
    BondFitTarget,
    CalibrationMode,
    CalibrationObjective,
    CalibrationSpec,
    CurveKernelKind,
    CurveSpec,
    CurveType,
    ExtrapolationPolicy,
    KernelSpec,
    YieldCurve,
)
from fuggers_py.rates import SwapQuote
from fuggers_py.vol_surfaces import VolPoint, VolQuoteType, VolSurfaceType, VolatilitySurface

REFERENCE_DATE = Date.from_ymd(2026, 4, 16)

curve = YieldCurve.fit(
    quotes=[
        SwapQuote(instrument_id="1Y", tenor="1Y", rate=0.025, currency="USD", as_of=REFERENCE_DATE),
        SwapQuote(instrument_id="2Y", tenor="2Y", rate=0.03, currency="USD", as_of=REFERENCE_DATE),
        SwapQuote(instrument_id="5Y", tenor="5Y", rate=0.035, currency="USD", as_of=REFERENCE_DATE),
    ],
    spec=CurveSpec(
        name="USD OIS",
        reference_date=REFERENCE_DATE,
        day_count="ACT/365F",
        currency="USD",
        type=CurveType.OVERNIGHT_DISCOUNT,
        extrapolation_policy=ExtrapolationPolicy.HOLD_LAST_ZERO_RATE,
    ),
    kernel_spec=KernelSpec(
        kind=CurveKernelKind.CUBIC_SPLINE,
        parameters={"knots": (1.0, 2.0, 5.0)},
    ),
    calibration_spec=CalibrationSpec(
        mode=CalibrationMode.GLOBAL_FIT,
        objective=CalibrationObjective.WEIGHTED_L2,
        bond_fit_target=BondFitTarget.DIRTY_PRICE,
    ),
)

surface = VolatilitySurface(
    surface_id="usd-swaption-grid",
    surface_type=VolSurfaceType.SWAPTION,
    as_of=REFERENCE_DATE,
    points=(
        VolPoint(
            expiry="2027-06",
            tenor="2032-06",
            volatility=Decimal("0.255"),
            strike=Decimal("0.03"),
            quote_type=VolQuoteType.NORMAL,
        ),
    ),
)

print(curve.rate_at(2.0))
print(surface.points[0].volatility)
